# Kaggle 제출 파일 생성

학습된 YOLO 모델로 테스트 이미지 추론 → `submission.csv` 생성

**평가 지표**: mAP@[0.75:0.95]  
**제출 형식**: CSV
```
annotation_id, image_id, category_id, bbox_x, bbox_y, bbox_w, bbox_h, score
1, 1, 1, 156, 247, 211, 456, 0.91
2, 1, 24, 498, 40, 460, 474, 0.78
...
```

## 1. 환경 설정

In [ ]:
from utils.sys_utils import add_experiment_root

PROJECT_ROOT = add_experiment_root(2)

In [ ]:
from pathlib import Path

In [ ]:
from src.inference import run_inference, save_submission

EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
KAGGLE_DATA = (
    Path.home()
    / ".cache/kagglehub/competitions/ai12-level1-project/sprint_ai_project1_data"
)
# KAGGLE_DATA = (
#     Path.cwd().parent
#     / "input"
#     / "competitions"
#     / "ai12-level1-project"
#     / "sprint_ai_project1_data"
# )

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"테스트 이미지: {KAGGLE_DATA / 'test_images'}")

## 2. 모델 선택

학습된 실험 폴더에서 `best.pt`와 `class_map.json`을 지정합니다.

In [ ]:
# 사용 가능한 실험 목록 확인
import json

print("사용 가능한 실험:")
for exp_dir in sorted(EXPERIMENTS_DIR.iterdir()):
    best_pt = exp_dir / "weights" / "best.pt"
    class_map = (
        exp_dir.parent.parent
        / "data"
        / "joelchoi"
        / f"yolo_{exp_dir.name.split('_')[2] if len(exp_dir.name.split('_')) > 2 else 'small'}"
        / "class_map.json"
    )
    metrics_f = exp_dir / "metrics.json"

    if best_pt.exists():
        metrics = json.loads(metrics_f.read_text()) if metrics_f.exists() else {}
        map_score = metrics.get("map75_95", metrics.get("map", 0))
        print(f"  [{exp_dir.name}]")
        print(f"    가중치: {best_pt}")
        print(f"    mAP@75-95: {map_score:.4f}")

In [ ]:
# ============================================================
# 사용할 실험 설정 (아래 값을 변경하세요)
# ============================================================
EXP_NAME = "exp001_yolo11n_tiny"  # 실험 폴더 이름
DATA_TIER = "small"  # 데이터 티어 (tiny / small / medium / full / kaggle)
CONF_THRESHOLD = 0.25  # confidence 임계값
IMG_SIZE = 640  # 추론 이미지 크기
# ============================================================

WEIGHTS = EXPERIMENTS_DIR / EXP_NAME / "weights" / "best.pt"
CLASS_MAP = PROJECT_ROOT / "data" / "joelchoi" / f"yolo_{DATA_TIER}" / "class_map.json"

print(f"가중치: {WEIGHTS}  존재: {WEIGHTS.exists()}")
print(f"클래스맵: {CLASS_MAP}  존재: {CLASS_MAP.exists()}")

## 3. 클래스 매핑 확인

> `class_map.json`이 없으면 아래 셀을 실행해서 생성합니다.

In [ ]:
if CLASS_MAP.exists():
    cm = json.loads(CLASS_MAP.read_text(encoding="utf-8"))
    print(f"클래스 수: {len(cm)}")
    for idx, info in list(cm.items())[:5]:
        print(f"  YOLO {idx} → category_id {info['category_id']}: {info['name']}")
    print("  ...")
else:
    print("⚠️  class_map.json 없음 → 아래 셀로 생성")

In [ ]:
# ──────────────────────────────────────────────────────────────
# [A] AI Hub 데이터로 학습한 경우: class_map.json 재생성
# class_map.json이 없거나 Kaggle category_id로 변환이 필요할 때 실행
# ──────────────────────────────────────────────────────────────
from src.data.aihub_converter import convert_aihub_to_coco
from src.data.subset import create_subset
from src.data.coco_to_yolo import prepare_yolo_dataset

DATA_DIR = PROJECT_ROOT / "data" / "joelchoi"

# 학습 시와 동일한 combo_nums, tier로 재생성
# coco = convert_aihub_to_coco(combo_nums=[1, 3, 4, 5, 6])
# train_coco, val_coco = create_subset(coco, tier=DATA_TIER, seed=42)
# yaml_path = prepare_yolo_dataset(train_coco, val_coco, DATA_DIR / f"yolo_{DATA_TIER}")
# print(f"class_map.json 생성됨: {DATA_DIR / f'yolo_{DATA_TIER}' / 'class_map.json'}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# [B] Kaggle 대회 데이터로 학습한 경우: class_map.json 생성
# (56 카테고리, 올바른 category_id 사용)
# ──────────────────────────────────────────────────────────────
from src.data.kaggle_converter import convert_kaggle_to_coco
from src.data.subset import create_subset
from src.data.coco_to_yolo import prepare_yolo_dataset

DATA_DIR = PROJECT_ROOT / "data" / "joelchoi"

# coco = convert_kaggle_to_coco()
# train_coco, val_coco = create_subset(coco, tier="full", seed=42)
# yaml_path = prepare_yolo_dataset(train_coco, val_coco, DATA_DIR / "yolo_kaggle")
# CLASS_MAP = DATA_DIR / "yolo_kaggle" / "class_map.json"
# print(f"class_map.json 생성됨: {CLASS_MAP}")

## 4. 추론 실행

In [ ]:
predictions = run_inference(
    weights_path=WEIGHTS,
    class_map_path=CLASS_MAP,
    test_dir=KAGGLE_DATA / "test_images",
    conf_threshold=CONF_THRESHOLD,
    iou_threshold=0.45,
    img_size=IMG_SIZE,
    batch_size=32,
)
print("\n샘플 예측:")
for p in predictions[:3]:
    print(f"  {p}")

## 5. 제출 파일 저장

In [ ]:
submission_path = SUBMISSION_DIR / f"{EXP_NAME}_submission.csv"
save_submission(predictions, submission_path)
print(f"\n제출 파일: {submission_path}")

## 6. 제출 파일 검증

In [ ]:
import pandas as pd

df = pd.read_csv(submission_path)

test_ids = {int(p.stem) for p in (KAGGLE_DATA / "test_images").glob("*.png")}
pred_ids = set(df["image_id"].unique()) if not df.empty else set()
missing = test_ids - pred_ids

print(f"전체 예측 수 (행): {len(df)}")
print(f"예측된 이미지: {len(pred_ids)} / {len(test_ids)}")
print(f"예측 없는 이미지: {len(missing)}개 (검출 없음으로 처리됨)")
print(
    f"사용된 category_id: {sorted(df['category_id'].unique().tolist()) if not df.empty else []}"
)
print()
if not df.empty:
    print(df.head(10).to_string(index=False))